# Pendahuluan
Notebook ini membahas proses agregasi data kecelakaan lalu lintas menggunakan Apache Spark (PySpark) berdasarkan dataset yang telah dibersihkan pada tahap sebelumnya. Agregasi dilakukan untuk mengetahui distribusi kecelakaan berdasarkan waktu, tingkat keparahan cedera, jenis kecelakaan, kondisi lingkungan, dan penyebab kecelakaan, yang selanjutnya digunakan sebagai dasar visualisasi data.

1. upload file data clean

In [5]:
from google.colab import files
uploaded = files.upload()

Saving data-cleaned_traffic_accidents_clean.zip to data-cleaned_traffic_accidents_clean.zip


2. pemeriksaan file

In [7]:
import os

print("Isi folder /content:")
print(os.listdir('/content'))

Isi folder /content:
['.config', 'data-cleaned_traffic_accidents_clean.zip', 'sample_data']


3. ekstrak file zip

In [8]:
import zipfile
import os

zip_path = '/content/data-cleaned_traffic_accidents_clean.zip'
extract_path = '/content/traffic_accidents'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Berhasil diekstrak!")
print("Isi folder:", os.listdir(extract_path))

Berhasil diekstrak!
Isi folder: ['data-cleaned_traffic_accidents_clean.csv', 'data-cleaned_traffic_accidents_clean.parquet']


4. periksa file parquet

In [9]:
for root, dirs, files in os.walk('/content/traffic_accidents'):
    for file in files:
        print(os.path.join(root, file))

/content/traffic_accidents/data-cleaned_traffic_accidents_clean.csv/part-00004-9fe4225b-19d8-41ab-ad0a-877717bbff80-c000.csv
/content/traffic_accidents/data-cleaned_traffic_accidents_clean.csv/part-00000-9fe4225b-19d8-41ab-ad0a-877717bbff80-c000.csv
/content/traffic_accidents/data-cleaned_traffic_accidents_clean.csv/part-00006-9fe4225b-19d8-41ab-ad0a-877717bbff80-c000.csv
/content/traffic_accidents/data-cleaned_traffic_accidents_clean.csv/part-00007-9fe4225b-19d8-41ab-ad0a-877717bbff80-c000.csv
/content/traffic_accidents/data-cleaned_traffic_accidents_clean.csv/part-00003-9fe4225b-19d8-41ab-ad0a-877717bbff80-c000.csv
/content/traffic_accidents/data-cleaned_traffic_accidents_clean.csv/part-00002-9fe4225b-19d8-41ab-ad0a-877717bbff80-c000.csv
/content/traffic_accidents/data-cleaned_traffic_accidents_clean.csv/part-00005-9fe4225b-19d8-41ab-ad0a-877717bbff80-c000.csv
/content/traffic_accidents/data-cleaned_traffic_accidents_clean.csv/part-00001-9fe4225b-19d8-41ab-ad0a-877717bbff80-c000.csv


5. membaca dataset parquet dengan PySpark

In [10]:
!pip install pyspark

In [11]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("TrafficAccidentAnalysis") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

print("Spark berhasil dijalankan!")

Spark berhasil dijalankan!


6. membaca data parquet

In [12]:
PARQUET_PATH = "/content/traffic_accidents/data-cleaned_traffic_accidents_clean.parquet"

df = spark.read.parquet(PARQUET_PATH)

print("Data berhasil dibaca!")
print("Jumlah baris:", df.count())
print("Jumlah kolom:", len(df.columns))

Data berhasil dibaca!
Jumlah baris: 209275
Jumlah kolom: 24


In [13]:
df.printSchema()
df.show(5, truncate=False)

root
 |-- crash_date: timestamp (nullable = true)
 |-- traffic_control_device: string (nullable = true)
 |-- weather_condition: string (nullable = true)
 |-- lighting_condition: string (nullable = true)
 |-- first_crash_type: string (nullable = true)
 |-- trafficway_type: string (nullable = true)
 |-- alignment: string (nullable = true)
 |-- roadway_surface_cond: string (nullable = true)
 |-- road_defect: string (nullable = true)
 |-- crash_type: string (nullable = true)
 |-- intersection_related_i: string (nullable = true)
 |-- damage: string (nullable = true)
 |-- prim_contributory_cause: string (nullable = true)
 |-- num_units: integer (nullable = true)
 |-- most_severe_injury: string (nullable = true)
 |-- injuries_total: double (nullable = true)
 |-- injuries_fatal: double (nullable = true)
 |-- injuries_incapacitating: double (nullable = true)
 |-- injuries_non_incapacitating: double (nullable = true)
 |-- injuries_reported_not_evident: double (nullable = true)
 |-- injuries_no_i

7. pemeriksaan rentang tanggal

In [14]:
from pyspark.sql.functions import min, max

df.select(
    min("crash_date").alias("Tanggal Awal"),
    max("crash_date").alias("Tanggal Akhir")
).show()

+-------------------+-------------------+
|       Tanggal Awal|      Tanggal Akhir|
+-------------------+-------------------+
|2013-03-03 09:48:00|2025-01-17 17:17:00|
+-------------------+-------------------+



# 8. AGGREGATION

9. Agregasi Jumlah Kecelakaan Berdasarkan Tahun

In [15]:
from pyspark.sql.functions import year, count, col

# Agregasi jumlah kecelakaan berdasarkan tahun
accidents_by_year = (
    df.withColumn("year", year(col("crash_date")))
      .groupBy("year")
      .agg(count("*").alias("total_accidents"))
      .orderBy("year")
)

accidents_by_year.show(20)

+----+---------------+
|year|total_accidents|
+----+---------------+
|2013|              2|
|2015|           1921|
|2016|           7625|
|2017|          17668|
|2018|          27437|
|2019|          27947|
|2020|          22778|
|2021|          25696|
|2022|          25539|
|2023|          25436|
|2024|          26215|
|2025|           1011|
+----+---------------+



10. Agregasi Jumlah kecelakaan berdasarkan bulan

In [16]:
from pyspark.sql.functions import col, count

accidents_by_month = (
    df.groupBy("crash_month")
      .agg(count("*").alias("total_accidents"))
      .orderBy("crash_month")
)

accidents_by_month.show()

+-----------+---------------+
|crash_month|total_accidents|
+-----------+---------------+
|          1|          16605|
|          2|          14617|
|          3|          15264|
|          4|          15095|
|          5|          17429|
|          6|          17848|
|          7|          17831|
|          8|          18346|
|          9|          19013|
|         10|          20088|
|         11|          18326|
|         12|          18813|
+-----------+---------------+



11. Agregasi berdasarkan tingkat keparahan cedera

In [17]:
from pyspark.sql.functions import count, col

accidents_by_injury = (
    df.groupBy("most_severe_injury")
      .agg(count("*").alias("total_accidents"))
      .orderBy(col("total_accidents").desc())
)

accidents_by_injury.show(truncate=False)

+------------------------+---------------+
|most_severe_injury      |total_accidents|
+------------------------+---------------+
|NO INDICATION OF INJURY |154767         |
|NONINCAPACITATING INJURY|31521          |
|REPORTED, NOT EVIDENT   |16073          |
|INCAPACITATING INJURY   |6563           |
|FATAL                   |351            |
+------------------------+---------------+



12. Agregasi berdasarkan jenis kecelakaan

In [18]:
accidents_by_type = (
    df.groupBy("crash_type")
      .agg(count("*").alias("total_accidents"))
      .orderBy(col("total_accidents").desc())
)

accidents_by_type.show(truncate=False)

+--------------------------------+---------------+
|crash_type                      |total_accidents|
+--------------------------------+---------------+
|NO INJURY / DRIVE AWAY          |117356         |
|INJURY AND / OR TOW DUE TO CRASH|91919          |
+--------------------------------+---------------+



13. Agregasi berdasarkan kondisi cuaca

In [19]:
accidents_by_weather = (
    df.groupBy("weather_condition")
      .agg(count("*").alias("total_accidents"))
      .orderBy(col("total_accidents").desc())
)

accidents_by_weather.show(truncate=False)

+------------------------+---------------+
|weather_condition       |total_accidents|
+------------------------+---------------+
|CLEAR                   |164673         |
|RAIN                    |21702          |
|CLOUDY/OVERCAST         |7532           |
|SNOW                    |6869           |
|UNKNOWN                 |6534           |
|OTHER                   |627            |
|FREEZING RAIN/DRIZZLE   |510            |
|FOG/SMOKE/HAZE          |360            |
|SLEET/HAIL              |308            |
|BLOWING SNOW            |127            |
|SEVERE CROSS WIND GATE  |32             |
|BLOWING SAND, SOIL, DIRT|1              |
+------------------------+---------------+



14. Agregasi berdasarkan kondisi permukaan jalan

In [20]:
accidents_by_road_surface = (
    df.groupBy("roadway_surface_cond")
      .agg(count("*").alias("total_accidents"))
      .orderBy(col("total_accidents").desc())
)

accidents_by_road_surface.show(truncate=False)

+--------------------+---------------+
|roadway_surface_cond|total_accidents|
+--------------------+---------------+
|DRY                 |155878         |
|WET                 |32906          |
|UNKNOWN             |12509          |
|SNOW OR SLUSH       |6201           |
|ICE                 |1303           |
|OTHER               |438            |
|SAND, MUD, DIRT     |40             |
+--------------------+---------------+



15. Agregasi berdasarkan jam kejadian

In [21]:
accidents_by_hour = (
    df.groupBy("crash_hour")
      .agg(count("*").alias("total_accidents"))
      .orderBy("crash_hour")
)

accidents_by_hour.show(24)

+----------+---------------+
|crash_hour|total_accidents|
+----------+---------------+
|         0|           4487|
|         1|           3729|
|         2|           3080|
|         3|           2364|
|         4|           2104|
|         5|           2819|
|         6|           4771|
|         7|           9405|
|         8|          11379|
|         9|           9298|
|        10|           8989|
|        11|          10050|
|        12|          11725|
|        13|          11947|
|        14|          13346|
|        15|          15890|
|        16|          16020|
|        17|          16095|
|        18|          13429|
|        19|           9682|
|        20|           8219|
|        21|           7470|
|        22|           7035|
|        23|           5942|
+----------+---------------+



16. Agregasi berdasarkan hari dalam seminggu

In [22]:
accidents_by_day = (
    df.groupBy("crash_day_of_week")
      .agg(count("*").alias("total_accidents"))
      .orderBy("crash_day_of_week")
)

accidents_by_day.show()

+-----------------+---------------+
|crash_day_of_week|total_accidents|
+-----------------+---------------+
|                1|          25245|
|                2|          27930|
|                3|          30068|
|                4|          30089|
|                5|          30784|
|                6|          34453|
|                7|          30706|
+-----------------+---------------+



17. Agregasi berdasarkan penyebab utama kecelakaan

In [23]:
accidents_by_cause = (
    df.groupBy("prim_contributory_cause")
      .agg(count("*").alias("total_accidents"))
      .orderBy(col("total_accidents").desc())
)

accidents_by_cause.show(30, truncate=False)

+--------------------------------------------------------------------------------+---------------+
|prim_contributory_cause                                                         |total_accidents|
+--------------------------------------------------------------------------------+---------------+
|UNABLE TO DETERMINE                                                             |58310          |
|FAILING TO YIELD RIGHT-OF-WAY                                                   |42909          |
|FOLLOWING TOO CLOSELY                                                           |19074          |
|DISREGARDING TRAFFIC SIGNALS                                                    |14590          |
|IMPROPER TURNING/NO SIGNAL                                                      |12642          |
|FAILING TO REDUCE SPEED TO AVOID CRASH                                          |10675          |
|IMPROPER OVERTAKING/PASSING                                                     |8302           |
|DISREGARD

18. Menyimpan seluruh hasil agregasi (csv dan parquet)

In [26]:
import os

AGGREGATED_DIR = "/content/traffic_accidents/aggregated"

os.makedirs(AGGREGATED_DIR, exist_ok=True)

print("Folder berhasil dibuat:", os.path.exists(AGGREGATED_DIR))

Folder berhasil dibuat: True


In [27]:
aggregations = {
    "accidents_by_year": accidents_by_year,
    "accidents_by_month": accidents_by_month,
    "accidents_by_injury": accidents_by_injury,
    "accidents_by_type": accidents_by_type,
    "accidents_by_weather": accidents_by_weather,
    "accidents_by_road_surface": accidents_by_road_surface,
    "accidents_by_hour": accidents_by_hour,
    "accidents_by_day": accidents_by_day,
    "accidents_by_cause": accidents_by_cause
}

for name, data in aggregations.items():

    # Simpan Parquet
    data.write.mode("overwrite").parquet(
        f"{AGGREGATED_DIR}/{name}.parquet"
    )

    # Simpan CSV
    data.coalesce(1).write.mode("overwrite") \
        .option("header", True) \
        .csv(f"{AGGREGATED_DIR}/{name}.csv")

    print(f"Berhasil menyimpan {name} dalam Parquet dan CSV")

print("Semua agregasi berhasil disimpan!")

Berhasil menyimpan accidents_by_year dalam Parquet dan CSV
Berhasil menyimpan accidents_by_month dalam Parquet dan CSV
Berhasil menyimpan accidents_by_injury dalam Parquet dan CSV
Berhasil menyimpan accidents_by_type dalam Parquet dan CSV
Berhasil menyimpan accidents_by_weather dalam Parquet dan CSV
Berhasil menyimpan accidents_by_road_surface dalam Parquet dan CSV
Berhasil menyimpan accidents_by_hour dalam Parquet dan CSV
Berhasil menyimpan accidents_by_day dalam Parquet dan CSV
Berhasil menyimpan accidents_by_cause dalam Parquet dan CSV
Semua agregasi berhasil disimpan!


In [28]:
import shutil
from google.colab import files

shutil.make_archive(
    "/content/hasil_agregasi_traffic_accidents",
    "zip",
    AGGREGATED_DIR
)

files.download(
    "/content/hasil_agregasi_traffic_accidents.zip"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

19. gabungkan dan rapikan file csv

In [29]:
import os
import shutil

SOURCE_DIR = "/content/traffic_accidents/aggregated"
CLEAN_CSV_DIR = "/content/hasil_agregasi_csv"

os.makedirs(CLEAN_CSV_DIR, exist_ok=True)

for name in aggregations.keys():

    folder = os.path.join(SOURCE_DIR, f"{name}.csv")

    for file in os.listdir(folder):
        if file.startswith("part-") and file.endswith(".csv"):

            source_file = os.path.join(folder, file)
            destination_file = os.path.join(
                CLEAN_CSV_DIR, f"{name}.csv"
            )

            shutil.copy2(source_file, destination_file)
            print(f"Berhasil: {name}.csv")

print("\nSemua CSV berhasil dirapikan!")

Berhasil: accidents_by_year.csv
Berhasil: accidents_by_month.csv
Berhasil: accidents_by_injury.csv
Berhasil: accidents_by_type.csv
Berhasil: accidents_by_weather.csv
Berhasil: accidents_by_road_surface.csv
Berhasil: accidents_by_hour.csv
Berhasil: accidents_by_day.csv
Berhasil: accidents_by_cause.csv

Semua CSV berhasil dirapikan!


In [30]:
print(os.listdir(CLEAN_CSV_DIR))

['accidents_by_day.csv', 'accidents_by_cause.csv', 'accidents_by_weather.csv', 'accidents_by_road_surface.csv', 'accidents_by_type.csv', 'accidents_by_hour.csv', 'accidents_by_year.csv', 'accidents_by_month.csv', 'accidents_by_injury.csv']


In [31]:
import shutil
from google.colab import files

shutil.make_archive(
    "/content/hasil_agregasi_csv",
    "zip",
    CLEAN_CSV_DIR
)

files.download("/content/hasil_agregasi_csv.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

20. gabungkan dan rapikan file parquet

In [32]:
import os
import shutil

CLEAN_PARQUET_DIR = "/content/hasil_agregasi_parquet"

os.makedirs(CLEAN_PARQUET_DIR, exist_ok=True)

for name, data in aggregations.items():

    temp_path = f"/content/temp_{name}"

    data.coalesce(1).write.mode("overwrite").parquet(temp_path)

    for file in os.listdir(temp_path):
        if file.endswith(".parquet") and file.startswith("part-"):

            shutil.copy2(
                os.path.join(temp_path, file),
                os.path.join(CLEAN_PARQUET_DIR, f"{name}.parquet")
            )

    shutil.rmtree(temp_path)

    print(f"Berhasil: {name}.parquet")

print("\nSemua Parquet berhasil dirapikan!")

Berhasil: accidents_by_year.parquet
Berhasil: accidents_by_month.parquet
Berhasil: accidents_by_injury.parquet
Berhasil: accidents_by_type.parquet
Berhasil: accidents_by_weather.parquet
Berhasil: accidents_by_road_surface.parquet
Berhasil: accidents_by_hour.parquet
Berhasil: accidents_by_day.parquet
Berhasil: accidents_by_cause.parquet

Semua Parquet berhasil dirapikan!


In [33]:
print(os.listdir(CLEAN_PARQUET_DIR))

['accidents_by_hour.parquet', 'accidents_by_cause.parquet', 'accidents_by_road_surface.parquet', 'accidents_by_day.parquet', 'accidents_by_injury.parquet', 'accidents_by_month.parquet', 'accidents_by_weather.parquet', 'accidents_by_type.parquet', 'accidents_by_year.parquet']


In [34]:
import shutil
from google.colab import files

shutil.make_archive(
    "/content/hasil_agregasi_parquet",
    "zip",
    CLEAN_PARQUET_DIR
)

files.download("/content/hasil_agregasi_parquet.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

21. validasi hasil agregasi


Validasi dilakukan untuk memastikan jumlah data pada setiap hasil agregasi sesuai dengan jumlah baris dataset awal.

In [35]:
total_data = df.count()

print("Total data awal:", total_data)
print("\nValidasi hasil agregasi:")

for name, data in aggregations.items():
    total_agregasi = data.agg(
        {"total_accidents": "sum"}
    ).collect()[0][0]

    status = "SESUAI" if total_agregasi == total_data else "TIDAK SESUAI"

    print(f"{name}: {total_agregasi} - {status}")

Total data awal: 209275

Validasi hasil agregasi:
accidents_by_year: 209275 - SESUAI
accidents_by_month: 209275 - SESUAI
accidents_by_injury: 209275 - SESUAI
accidents_by_type: 209275 - SESUAI
accidents_by_weather: 209275 - SESUAI
accidents_by_road_surface: 209275 - SESUAI
accidents_by_hour: 209275 - SESUAI
accidents_by_day: 209275 - SESUAI
accidents_by_cause: 209275 - SESUAI


# Kesimpulan
Proses agregasi berhasil dilakukan terhadap 209.275 data kecelakaan lalu lintas menggunakan Apache Spark (PySpark). Hasil pengelompokan berdasarkan sembilan atribut telah disimpan dalam format CSV dan Parquet untuk digunakan pada tahap visualisasi data di Notebook 04.